# Book Recommendation System
Modified version using the popular Kaggle Book Recommendation Dataset

## 1. Download the Dataset from Kaggle

In [ ]:
# Option 1: Download manually from https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset
# Then extract the CSV files in your current directory

# Option 2: Use Kaggle API (if you have it set up)
# ! pip install kaggle
#! kaggle datasets download -d arashnic/book-recommendation-dataset
# ! unzip book-recommendation-dataset.zip

print("✓ Dataset files should be in current directory:")
print("  - books.csv")
print("  - ratings.csv")
print("  - users.csv")

## 2. Setup and Imports

In [ ]:
%pip install -q langchain langchain-core langchain-openai langchain-google-genai langchain-huggingface langchain-community python-dotenv pandas pydantic

import os
import json
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv("../.env")

print("✓ Base imports successful")

## 3. Load and Explore Real Dataset from Kaggle

In [ ]:
# Load the Kaggle datasets
print("Loading Kaggle Book Recommendation Dataset...\n")

books_df = pd.read_csv('data/Books.csv', encoding='latin-1')
ratings_df = pd.read_csv('data/Ratings.csv')
users_df = pd.read_csv('data/Users.csv')

print(f"Books dataset: {books_df.shape}")
print(f"Ratings dataset: {ratings_df.shape}")
print(f"Users dataset: {users_df.shape}")

# Display sample data
print("\n=== Books Dataset Sample ===")
print(books_df.head(3))

print("\n=== Ratings Dataset Sample ===")
print(ratings_df.head(3))

print("\n=== Books Dataset Columns ===")
print(books_df.columns.tolist())

## 4. Prepare Data for LangChain (Sample for Speed)

In [ ]:
# Sample the data to make processing faster
# The full dataset has 271K books - we'll use top books by ratings

# Merge ratings with books
#merged = ratings_df.merge(books_df, on='ISBN', how='inner')

# Get top books by number of ratings
top_books_isbn = ratings_df.groupby('ISBN').size().nlargest(1000).index
print(f"Selecting top 1000 books by rating count...")

# Filter to top books
books_sample = books_df[books_df['ISBN'].isin(top_books_isbn)].copy()

# Rename columns to be more user-friendly
books_sample.rename(columns={
    'Book-Title': 'title',
    'Book-Author': 'author',
    'Year-Of-Publication': 'year',
    'Publisher': 'publisher',
    'ISBN': 'isbn'
}, inplace=True)

# Save sampled dataset for document loader
books_sample.to_csv('books_for_llm.csv', index=False)

print(f"✓ Sampled dataset: {books_sample.shape[0]} books")
print(f"\nDataset Preview:")
print(books_sample[['title', 'author', 'year', 'publisher', 'isbn']].head(10))

## 5. Model Initialization (Model Agnostic)

In [ ]:
from langchain.chat_models import init_chat_model

# Initialize a model-agnostic LLM
# Set OPENAI_API_KEY environment variable with your OpenAI API key

#OpenAI "openai:gpt-4o-mini" or "openai:gpt-4"
#Google Gemini "google_genai:gemini-2.5-flash-lite"
#HuggingFace "huggingface_hub:meta-llama/Llama-2-7b-chat-hf"

llm = init_chat_model(
    model="openai:gpt-4o-mini",  # Free trial credits available
    temperature=0.7,
    max_tokens=500
)

print("✓ LLM initialized successfully")

## 6. Messages - Simple Conversation

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# Messages provide a structured way to manage conversations
messages = [
    SystemMessage(content="You are a knowledgeable book recommendation assistant with expertise in literature."),
    HumanMessage(content="Recommed some classic literature books for me to read.")
]

response = llm.invoke(messages)
print("Assistant:", response.content[:500])
print("\n✓ Message-based conversation working")

## 7. Document Loader - Load Books Sample Data

In [ ]:
from langchain_community.document_loaders import CSVLoader

# Load books sample with LangChain CSVLoader (creates Document objects directly)
#  Each document represents one row of the CSV file
loader = CSVLoader(file_path="books_for_llm.csv")
documents = loader.load()

print(f"✓ Loaded {len(documents)} book documents for LLM context")
print(f"\nFirst book document:")
print(documents[0].page_content)

## 8. Inject Documents into Prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Convert loaded documents to formatted text
books_context = "\n\n".join([doc.page_content for doc in documents])

print(f"Books context length: {len(books_context)} characters")
print(f"\nSample context (first 500 chars):\n{books_context[:500]}")

# Create a prompt template with books_context hardcoded
recommendation_prompt = ChatPromptTemplate.from_template(
    "You are a book recommendation expert with knowledge of classic and contemporary literature.\n"
    "Here are available books from our catalog:\n"
    + books_context +
    "\nUser request: {user_query}\n"
    "Provide thoughtful recommendations based on the user's preference."
)

# Format the prompt with user query only
formatted_prompt = recommendation_prompt.format(
    user_query="I enjoy fiction books from the 1990s and 2000s"
)

response = llm.invoke(formatted_prompt)

print(response.content)


## 9. Basic LangChain Expression Language (LCEL) Chaining

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Basic LCEL chain: Prompt → LLM → Output

simple_chain = recommendation_prompt | llm | StrOutputParser()

result = simple_chain.invoke({
    "user_query": "I want books by popular American authors from the 1980s"
})

print("Chain Output (from Real Dataset):")
print(result)


## 10. Control Flow - RunnableBranch

In [ ]:
from langchain_core.runnables import RunnableBranch

# Create different chains for different intents
recommendation_chain = (
    recommendation_prompt | llm | StrOutputParser()
)

analysis_chain = (
    ChatPromptTemplate.from_template(
        f"You are a book expert. Catalog books: {books_context}\n\n"
        "Analyze/discuss: {user_query}"
    )
    | llm
    | StrOutputParser()
)

# Create branching logic
router = RunnableBranch(
    (
        lambda x: "recommend" in x["user_query"].lower(),
        recommendation_chain
    ),
    (
        lambda x: "compare" in x["user_query"].lower() or "analysis" in x["user_query"].lower(),
        analysis_chain
    ),
    recommendation_chain  # default
)

# Test with different queries
print("Test 1 - Recommend query:")
result1 = router.invoke({
    "user_query": "recommend a mystery thriller book"
})
print(result1[:300] + "...\n")

print("Test 2 - Analysis query:")
result2 = router.invoke({
    "user_query": "analyze trends in literature from 1990s"
})
print(result2[:300] + "...")
print("\n✓ RunnableBranch working")

## 11. Generate-Refine Pipeline

In [ ]:
# Step 1: Generate initial recommendations
recommend_chain = (
    recommendation_prompt | llm | StrOutputParser()
)

# Step 2: Refine the recommendations
refine_chain = (
    ChatPromptTemplate.from_template(
        "Here are some book recommendations:\n{recommendation}\n\n"
        "Please refine these recommendations by:\n"
        "1. Making them more concise\n"
        "2. Adding specific reasons why each book is perfect\n"
        "3. Mentioning the publication year\n"
        "4. Formatting them clearly"
    )
    | llm
    | StrOutputParser()
)

# Combine into a two-step pipeline
generate_refine_chain = (
    recommend_chain
    | (lambda x: {"recommendation": x})   #Takes the plain string output of recommend_chain and formats "recommendation: string" it for refine_chain
    | refine_chain
)

# Execute the pipeline
final_recommendation = generate_refine_chain.invoke({
    "user_query": "I like literary fiction and historical novels"
})

print("Refined Recommendation from Real Data:")
print(final_recommendation)


## 12. Message History Memory - Multi-turn Conversations

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Create an in-memory chat history store
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Create a conversational chain with message history
conversational_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        f"You are a book recommendation assistant with knowledge of our Kaggle catalog. Available books:\n{books_context}"
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_query}")
])

conversational_chain = (
    conversational_prompt
    | llm
    | StrOutputParser()
)

# Wrap with message history management
memory_chain = RunnableWithMessageHistory(
    conversational_chain,
    get_session_history,
    input_messages_key="user_query",
    history_messages_key="history"
)

session_id = "user_session_001"

# Turn 1
print("=== Turn 1 ===")
response1 = memory_chain.invoke(
    {
        "user_query": "I enjoy classic literature. Can you recommend something?"
    },
    config={"configurable": {"session_id": session_id}}
)
print(response1[:300] + "...")

# Turn 2 - Bot should remember previous context
print("\n=== Turn 2 (Bot remembers context) ===")
response2 = memory_chain.invoke(
    {
        "user_query": "Who wrote the first book you recommended?"
    },
    config={"configurable": {"session_id": session_id}}
)
print(response2[:300] + "...")

# View conversation history
print("\n=== Full Conversation History ===")
history = get_session_history(session_id)
for i, msg in enumerate(history.messages):
    print(f"{i+1}. [{msg.type.upper()}]: {msg.content[:80]}...")



## 13. Serialization - Save and Load Chains

In [ ]:
from langchain_core.load import dumps, loads

# Serialize the shared recommendation_prompt from Section 8
serialized = dumps(recommendation_prompt)
print("Serialized Prompt (JSON):")
print(serialized[:500] + "...")

# Save to file
with open("book_chain_kaggle.json", "w") as f:
    f.write(serialized)
print("\n✓ Prompt saved to 'book_chain_kaggle.json'")

# Load the prompt from file
with open("book_chain_kaggle.json", "r") as f:
    loaded_serialized = f.read()

loaded_prompt = loads(loaded_serialized, allowed_objects="all")
print("✓ Prompt loaded from file")

# Rebuild the executable chain with the active llm in memory
loaded_chain = loaded_prompt | llm | StrOutputParser()

# Test the loaded chain (hardcoded catalog is already inside recommendation_prompt)
result = loaded_chain.invoke({
    "user_query": "fantasy and adventure novels"
})
print(f"\nLoaded Chain Output: {result[:300]}...")


## Summary

✅ **Dataset**: Real Kaggle Book Recommendation Dataset (271K books, 278K users, 1.1M ratings)  
✅ **Processing**: Sampled to 1000 popular books for faster LLM processing  
✅ **All Concepts**: Messages, Document Loader, Injection, Parser, LCEL, Control Flow, Generate-Refine, Memory, Serialization  
✅ **Model Agnostic**: Easy to switch between OpenAI, Gemini, HuggingFace  

